In [27]:
import requests       # чтобы скачивать страницы из интернета
from bs4 import BeautifulSoup  # чтобы разбирать HTML и искать нужные элементы
import pandas as pd   # чтобы создать таблицу DataFrame
import time           # чтобы делать паузы между запросами
from urllib.parse import urljoin

Определить формат ссылки для пагинации - 1 балл

In [28]:
# Вместо {} Python подставит номер страницы.
pagination_template = "https://lifehacker.ru/topics/technology/page/{}/"

# Создаём пустой список для ссылок
pagination_urls = []

# range(1, 11) даёт числа: 1, 2, 3, ..., 10
for page in range(1, 11):

    # Подставляем номер страницы в шаблон
    url = pagination_template.format(page)

    # Добавляем готовую ссылку в список
    pagination_urls.append(url)

# Печатаем все ссылки, чтобы проверить результат
for url in pagination_urls:
    print(url)

https://lifehacker.ru/topics/technology/page/1/
https://lifehacker.ru/topics/technology/page/2/
https://lifehacker.ru/topics/technology/page/3/
https://lifehacker.ru/topics/technology/page/4/
https://lifehacker.ru/topics/technology/page/5/
https://lifehacker.ru/topics/technology/page/6/
https://lifehacker.ru/topics/technology/page/7/
https://lifehacker.ru/topics/technology/page/8/
https://lifehacker.ru/topics/technology/page/9/
https://lifehacker.ru/topics/technology/page/10/


В разметке страницы найти уникальные классы/идентификаторы блока с названием и содержанием материала- 3 балла

In [29]:
# Выбираем любую ссылку на статью для проверки
test_url = "https://lifehacker.ru/v-politike-messendzhera-max-poyavilsya-sbor-dannyx-o-podpiskax/"

# Заголовки нужны, чтобы сайт не подумал,
# что к нему обращается робот
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Скачиваем страницу
response = requests.get(test_url, headers=headers)

# Проверяем, что страница скачалась без ошибки
response.raise_for_status()

# Получаем HTML в виде текста
html = response.text

# Превращаем HTML в объект BeautifulSoup,
# с которым удобно работать
soup = BeautifulSoup(html, "html.parser")

# Ищем первый тег h1 — это обычно заголовок статьи
title_tag = soup.find("h1")

# Достаём из него текст
title = title_tag.get_text(strip=True)

print("Заголовок:")
print(title)

Заголовок:
В политике мессенджера Max появился сбор данных о подписках и отписках от каналов — его нельзя отключить


In [30]:
# Ищем основной блок статьи.
# Сначала пробуем тег article — он часто содержит весь текст.
content_tag = soup.find("article")

if content_tag:
    # Удаляем ненужные элементы
    for unwanted in content_tag.find_all(["script", "style", "button", "form"]):
        unwanted.decompose()

    # Находим все абзацы
    paragraphs = content_tag.find_all("p")

    # Собираем текст из всех абзацев
    text = "\n".join(
        paragraph.get_text(strip=True)
        for paragraph in paragraphs
    )

    print("\nНачало текста:")
    print(text[:500])
else:
    print("Не нашли блок article")


Начало текста:
МессенджерMaxобновилполитику конфиденциальности и отдельно прописал сбор данных о том, как пользователи взаимодействуют с каналами и контентом. В новой редакции документа, датированной 25 сентября, появились сведения о подписках и отписках от каналов.
✍️ Скоро читатели смогут публиковать свои тексты на Лайфхакере.Станьте первыми!
Такой информации не было в предыдущей редакции политики от 18 мая. При этомMaxи раньше обрабатывал данные об активности внутри сервиса — например, клики и переходы в ин


Получить содержимое десяти страниц списка материалов, выделить из него ссылки на каждый материал - 2 балла

In [31]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Адрес первой страницы.
first_page_url = "https://lifehacker.ru/topics/technology/"

# Шаблон страниц со второй по десятую.
pagination_template = "https://lifehacker.ru/topics/technology/?page={}"

# Новый пустой список для ссылок.
article_links = []

# Перебираем первые 10 страниц рубрики.
for page_number in range(1, 11):

    # Формируем адрес страницы.
    if page_number == 1:
        page_url = first_page_url
    else:
        page_url = pagination_template.format(page_number)

    print("\nОткрываем страницу:", page_number)
    print(page_url)

    # Скачиваем HTML страницы.
    response = requests.get(
        page_url,
        headers=headers,
        timeout=30
    )

    # Проверяем, что сервер не вернул HTTP-ошибку.
    response.raise_for_status()

    # Разбираем HTML.
    soup = BeautifulSoup(response.text, "html.parser")

    # Ищем ссылки по классу, который найден в вашей разметке.
    cards = soup.find_all(
        "a",
        class_="lh-small-article-card__link",
        href=True
    )

    print("Найдено ссылок карточек:", len(cards))

    # Если нужный класс не найден, не продолжаем молча.
    if len(cards) == 0:
        raise ValueError(
            f"На странице {page_number} не найдены ссылки карточек. "
            "Нужно проверить полученный HTML."
        )

    # Запоминаем количество ссылок до обработки страницы.
    count_before = len(article_links)

    # Перебираем найденные ссылки карточек.
    for card in cards:

        # Получаем адрес из атрибута href.
        href = card["href"]

        # Превращаем относительную ссылку в полную.
        # Например:
        # /nazvanie-statii/
        # →
        # https://lifehacker.ru/nazvanie-statii/
        article_url = urljoin(response.url, href)

        # Не добавляем одну и ту же ссылку повторно.
        if article_url not in article_links:
            article_links.append(article_url)

    print(
        "Добавлено уникальных ссылок:",
        len(article_links) - count_before
    )

    # Пауза перед следующим запросом.
    time.sleep(1)

# Итог.
print("\nВсего уникальных ссылок:", len(article_links))

print("\nПервые 10 ссылок:")

for link in article_links[:10]:
    print(link)


Открываем страницу: 1
https://lifehacker.ru/topics/technology/
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 2
https://lifehacker.ru/topics/technology/?page=2
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 3
https://lifehacker.ru/topics/technology/?page=3
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 4
https://lifehacker.ru/topics/technology/?page=4
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 5
https://lifehacker.ru/topics/technology/?page=5
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 6
https://lifehacker.ru/topics/technology/?page=6
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 7
https://lifehacker.ru/topics/technology/?page=7
Найдено ссылок карточек: 30
Добавлено уникальных ссылок: 30

Открываем страницу: 8
https://lifehacker.ru/topics/technology/?page=8
Найдено ссылок карт

Перебрать все полученные ссылки и получить html-код каждого материала - 1 балл

In [32]:
print(pagination_template.format(2))

https://lifehacker.ru/topics/technology/?page=2


In [33]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Берём только первые 10 ссылок из списка пункта 3.
links_to_download = article_links[:10]

# Создаём новый пустой словарь.
# Здесь будем хранить HTML скачанных страниц.
articles_html = {}

print("Ссылок для скачивания:", len(links_to_download))

# Перебираем только выбранные 10 ссылок.
for article_url in links_to_download:

    print("\nСкачиваем:", article_url)

    try:
        # Получаем страницу по ссылке.
        response = requests.get(
            article_url,
            headers=headers,
            timeout=30
        )

        # Проверяем, что сервер не вернул HTTP-ошибку.
        response.raise_for_status()

        # Сохраняем HTML в словарь.
        articles_html[article_url] = response.text

        print("HTML сохранён.")

    except requests.RequestException as error:
        print("Не удалось скачать страницу:", error)

    # Пауза перед следующим запросом.
    time.sleep(1)

print("\nСкачано страниц:", len(articles_html))

Ссылок для скачивания: 10

Скачиваем: https://lifehacker.ru/10-luchshih-kompyuterov-sdelannyh-v-sssr/
HTML сохранён.

Скачиваем: https://lifehacker.ru/nado-brat-televizor-ot-yandeksa-s-alisoi-43/
HTML сохранён.

Скачиваем: https://lifehacker.ru/smartfony-i-planshety-galaxy-kotorye-poluchat-android-17/
HTML сохранён.

Скачиваем: https://lifehacker.ru/nam-konec-nobelevskij-laureat-po-fizike-dzheffri-xinton-zayavil-chto-ii-uzhe-obryol-soznanie/
HTML сохранён.

Скачиваем: https://lifehacker.ru/predstavlena-domashnyaya-ferma-dlya-zeleni/
HTML сохранён.

Скачиваем: https://lifehacker.ru/budilnik-na-telefon/
HTML сохранён.

Скачиваем: https://lifehacker.ru/anons-canva-craft/
HTML сохранён.

Скачиваем: https://lifehacker.ru/anons-smartfona-coolpad-s-shirokim-ekranom/
HTML сохранён.

Скачиваем: https://lifehacker.ru/kak-podklyuchit-besprovodnuyu-myshku-k-noutbuku/
HTML сохранён.

Скачиваем: https://lifehacker.ru/apple-uzhe-testiruet-iphone-duo-2/
HTML сохранён.

Скачано страниц: 10


Распарсить полученный текст разметки с помощью BeautifulSoup, вытащив по каждой ссылке заголовок и содержание материала - 2 балл

In [35]:
# Создаём пустой список для результатов.
# Каждая статья будет сохранена отдельным словарём.
articles_data = []

print("Страниц для обработки:", len(articles_html))

# Перебираем словарь из пункта 4.
# article_url — ссылка на страницу.
# html — HTML-код этой страницы.
for article_url, html in articles_html.items():

    # Превращаем HTML в объект BeautifulSoup.
    soup = BeautifulSoup(html, "html.parser")

    # --------------------------------------------------
    # 1. Находим заголовок
    # --------------------------------------------------

    # Ищем первый тег h1.
    title_tag = soup.find("h1")

    # Проверяем, найден ли тег.
    if title_tag is not None:

        # Извлекаем текст заголовка.
        title = title_tag.get_text(separator=" ", strip=True)

    else:
        title = ""

    # --------------------------------------------------
    # 2. Находим блок с содержанием статьи
    # --------------------------------------------------

    # Это предварительный вариант поиска.
    # Если article не содержит нужный текст,
    # потребуется точный класс блока из разметки сайта.
    content_tag = soup.find("article")

    # Создаём список для текстовых частей статьи.
    text_parts = []

    if content_tag is not None:

        # Убираем технические и навигационные элементы
        # внутри найденного блока.
        for tag in content_tag.find_all(
            ["script", "style", "nav", "aside", "form", "button"]
        ):
            tag.decompose()

        # Находим абзацы и подзаголовки внутри блока.
        blocks = content_tag.find_all(["p", "h2", "h3"])

        # Перебираем найденные текстовые блоки.
        for block in blocks:

            # Получаем текст без HTML-тегов.
            block_text = block.get_text(separator=" ", strip=True)

            # Пустые строки не добавляем.
            if block_text != "":
                text_parts.append(block_text)

    # Соединяем части статьи переносами строк.
    text = "\n".join(text_parts)

    # --------------------------------------------------
    # 3. Сохраняем результат
    # --------------------------------------------------

    article = {
        "url": article_url,
        "title": title,
        "text": text
    }

    articles_data.append(article)

    # Показываем результат обработки текущей страницы.
    print("\nСсылка:", article_url)
    print("Заголовок:", title)
    #print("Символов в тексте:", len(text))
    print("Содержание:")
    print(text)
    print("\n" + "-"*100)

    if title == "":
        print("Внимание: заголовок не найден.")

    if text == "":
        print("Внимание: текст не найден. Нужно уточнить блок содержимого.")

# Итоговое количество обработанных страниц.
print("\nОбработано страниц:", len(articles_data))

Страниц для обработки: 10

Ссылка: https://lifehacker.ru/10-luchshih-kompyuterov-sdelannyh-v-sssr/
Заголовок: 10 персональных компьютеров, сделанных в СССР
Содержание:
Несмотря на то что очень немногие советские граждане могли позволить себе персональный компьютер, в 80‑е годы в СССР активно разрабатывали такие устройства. Было представлено много продуктов, а это — список 10 наиболее любопытных моделей.
1. Агат
Годы выпуска: 1984–1993.
Компьютер «Агат» был первым подобным устройством, созданным для широкого распространения и использования в системе школьного обучения. Его разработали на основе Apple II, направив в серийное производство в 1984 году. Интересно, что этот советский ПК выпускали и после распада СССР — вплоть до 1993 года.
Объём системного ПЗУ «Агата» ограничивался 2 КБ, а объём ОЗУ в базовой комплектации составлял 64 КБ, хотя мог доходить до 128 КБ — в зависимости от поколения и установленных модулей памяти. В комплекте также шли два игровых джойстика.
2. Корвет
Год выпуска

Создать датайфрейм с полученными данными - 1 балл

In [39]:
df = pd.DataFrame(
    articles_data,
    columns=["url", "title", "text"]
)
df

,url,title,text
0,https://lifehacker.ru/10-luchshih-kompyuterov-...,"10 персональных компьютеров, сделанных в СССР",Несмотря на то что очень немногие советские гр...
1,https://lifehacker.ru/nado-brat-televizor-ot-y...,Надо брать: 43-дюймовый телевизор от «Яндекса»...,Многие ставят рядом с телевизором умную колонк...
2,https://lifehacker.ru/smartfony-i-planshety-ga...,"Названы смартфоны и планшеты Samsung Galaxy, к...",Samsung опубликовала официальный список смартф...
3,https://lifehacker.ru/nam-konec-nobelevskij-la...,«Нам конец»: нобелевский лауреат по физике Дже...,Британский учёный и нобелевский лауреат Джеффр...
4,https://lifehacker.ru/predstavlena-domashnyaya...,Домашняя ферма Harvest позволяет собирать свеж...,Британский стартап Home Harvest представил на ...
5,https://lifehacker.ru/budilnik-na-telefon/,Не проспите: 10 лучших приложений-будильников ...,"«Ещё пять минуточек» — самая наглая ложь, кото..."
6,https://lifehacker.ru/anons-canva-craft/,"Canva выпустила генератор картинок, который сн...",Canva представила Craft — свою первую нейросет...
7,https://lifehacker.ru/anons-smartfona-coolpad-...,Представлен бюджетный аналог Huawei Pura X Vie...,Китайский бренд Coolpad подхватил тренд на шир...
8,https://lifehacker.ru/kak-podklyuchit-besprovo...,Объясняем за минуту: как подключить беспроводн...,Беспроводные мыши подключаются к ноутбуку или ...
9,https://lifehacker.ru/apple-uzhe-testiruet-iph...,Apple уже тестирует iPhone Duo 2 — с одним важ...,Apple уже работает над вторым поколением склад...
